# 📊 Code 2: Clean and Preprocess Data (V2 - Using Close Only)

## Purpose
Clean and preprocess the filtered dataset by handling **missing values**, **outliers**, and other **data quality issues**.

## Input
- **`data_raw.csv`** (from Code 1b)

## Output
- **`data_clean.csv`** - Cleaned and preprocessed dataset
- `cleaning_report.csv` - Detailed cleaning report

## What This Code Does:
1. Identify and handle missing values
2. Detect and handle outliers using **absolute thresholds** (±100%/-80%)
3. Fix data quality issues
4. Remove or impute invalid data points
5. Show before/after effects at each step

## Key Updates:
- **Using Close only** (no Adj_Close) - split-adjusted prices
- **Absolute threshold** outlier detection (±100%/-80%)
- All returns calculated using Close

---

**⏱️ Estimated Time:** 5-10 minutes

## Step 1: Import Libraries

In [1]:
import sys
!pip install xlsxwriter --quiet

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"📅 Script run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.4 MB/s eta 0:00:00
✅ Libraries imported successfully!
📅 Script run date: 2026-06-16 20:29:51


## Step 2: Configuration

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Input/Output files
INPUT_FILE = 'data_raw.csv'
OUTPUT_FILE = 'data_clean.csv'
CLEANING_REPORT_FILE = 'cleaning_report.csv'
CLEANING_REPORT_XLSX = 'cleaning_report.xlsx'  # NEW: detailed multi-tab report
TICKER_CLEANING_FILE = 'ticker_level_cleaning.csv'  # NEW: optional manual ticker cleaning

# Outlier thresholds - ABSOLUTE approach
UPPER_RETURN_THRESHOLD = 1.00  # +100% daily return
LOWER_RETURN_THRESHOLD = -0.80  # -80% daily return

# Missing value handling
MAX_MISSING_CONSECUTIVE = 5  # Maximum consecutive missing days before dropping

# Report thresholds (INDEPENDENT of outlier-removal thresholds above)
# These drive the extreme-move SCAN in the diagnostic + Excel report.
DIAG_UPPER_PCT = 50.0    # flag daily returns above this (%) in the report
DIAG_LOWER_PCT = -50.0   # flag daily returns below this (%) in the report
DIAG_CONTEXT_ROWS = 10   # rows before & after each flagged row

print("="*80)
print("CODE 2: DATA CLEANING CONFIGURATION (V2)")
print("="*80)
print(f"Input File: {INPUT_FILE}")
print(f"Output File: {OUTPUT_FILE}")
print(f"\nOutlier Detection: ABSOLUTE THRESHOLDS")
print(f"  - Upper threshold: +{UPPER_RETURN_THRESHOLD*100:.0f}% daily return")
print(f"  - Lower threshold: {LOWER_RETURN_THRESHOLD*100:.0f}% daily return")
print(f"\n💡 Using Close (split-adjusted) for all calculations")
print("="*80)

CODE 2: DATA CLEANING CONFIGURATION (V2)
Input File: data_raw.csv
Output File: data_clean.csv

Outlier Detection: ABSOLUTE THRESHOLDS
  - Upper threshold: +100% daily return
  - Lower threshold: -80% daily return

💡 Using Close (split-adjusted) for all calculations


## Step 3: Load Data

**⚠️ Make sure `data_raw.csv` (from Code 1b) is available.**

In [3]:
print("Loading data...")
print("-"*80)

# Load data
df = pd.read_csv(INPUT_FILE)
df['Date'] = pd.to_datetime(df['Date'])

# Store initial counts
initial_records = len(df)
initial_stocks = df['Ticker'].nunique()

print(f"✅ Data loaded successfully!")
print(f"\n📊 INITIAL DATASET:")
print(f"   Total records: {initial_records:,}")
print(f"   Unique stocks: {initial_stocks}")
print(f"   Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"   Columns: {list(df.columns)}")
print("-"*80)

Loading data...
--------------------------------------------------------------------------------
✅ Data loaded successfully!

📊 INITIAL DATASET:
   Total records: 2,449,347
   Unique stocks: 575
   Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Value_Traded', 'Value_Traded_Cr']
--------------------------------------------------------------------------------


## Step 3a-pre: Shared Helper Functions

Reusable functions for (1) scanning extreme moves & bad prices and (2) building ±N-row context windows. Used by the ticker-cleaning step, the diagnostic, and the final Excel report.

In [4]:
# ============================================================================
# SHARED HELPERS (used by diagnostic + Excel report)
# ============================================================================
import numpy as np

def scan_extremes(data, upper_pct, lower_pct):
    """Return a per-ticker summary + the flagged row mask/index for a dataframe.
    Flags: daily return (on Close) > upper_pct or < lower_pct, or Close NaN/<=0.
    `data` must contain Date, Ticker, Close (and ideally OHLCV)."""
    d = data.sort_values(['Ticker', 'Date']).reset_index(drop=True).copy()
    d['Diag_Return_Pct'] = d.groupby('Ticker')['Close'].pct_change() * 100

    ret_low   = d['Diag_Return_Pct'] < lower_pct
    ret_high  = d['Diag_Return_Pct'] > upper_pct
    price_nan = d['Close'].isna()
    price_bad = (d['Close'] <= 0)

    d['Flag_Reason'] = ''
    d.loc[ret_low,   'Flag_Reason'] += f'Return<{lower_pct}%; '
    d.loc[ret_high,  'Flag_Reason'] += f'Return>{upper_pct}%; '
    d.loc[price_nan, 'Flag_Reason'] += 'Close=NaN; '
    d.loc[price_bad, 'Flag_Reason'] += 'Close<=0; '

    flagged_mask = ret_low | ret_high | price_nan | price_bad

    # Per-ticker summary counts
    summary = d.groupby('Ticker').agg(
        Total_Rows=('Date', 'size'),
        Count_Return_Above=('Diag_Return_Pct', lambda s: int((s > upper_pct).sum())),
        Count_Return_Below=('Diag_Return_Pct', lambda s: int((s < lower_pct).sum())),
    ).reset_index()
    # NaN / bad price counts per ticker
    nan_cnt = d.assign(_nan=price_nan.astype(int)).groupby('Ticker')['_nan'].sum()
    bad_cnt = d.assign(_bad=price_bad.astype(int)).groupby('Ticker')['_bad'].sum()
    summary['Count_Close_NaN'] = summary['Ticker'].map(nan_cnt).fillna(0).astype(int)
    summary['Count_Close_LE0'] = summary['Ticker'].map(bad_cnt).fillna(0).astype(int)
    summary['Total_Flagged'] = (summary['Count_Return_Above'] + summary['Count_Return_Below']
                                + summary['Count_Close_NaN'] + summary['Count_Close_LE0'])
    # Keep only tickers with at least one flag, sorted by most flags
    summary = summary[summary['Total_Flagged'] > 0].sort_values(
        'Total_Flagged', ascending=False).reset_index(drop=True)

    return d, flagged_mask, summary


def build_context(d, flagged_mask, context_rows):
    """Given the sorted frame `d` and its flagged mask, return ±context_rows
    windows around each flagged row (never crossing ticker boundaries)."""
    flagged_idx = d.index[flagged_mask].tolist()
    if len(flagged_idx) == 0:
        return pd.DataFrame()

    ticker_bounds = d.groupby('Ticker').apply(
        lambda g: (g.index.min(), g.index.max())).to_dict()

    rows_to_keep = set()
    case_map = {}
    for cid, ridx in enumerate(flagged_idx, start=1):
        tkr = d.at[ridx, 'Ticker']
        lo, hi = ticker_bounds[tkr]
        start = max(lo, ridx - context_rows)
        end   = min(hi, ridx + context_rows)
        for r in range(start, end + 1):
            rows_to_keep.add(r)
            case_map.setdefault(r, []).append(cid)

    out = d.loc[sorted(rows_to_keep)].copy()
    out['Case_IDs'] = out.index.map(lambda r: ','.join(map(str, case_map.get(r, []))))
    out['Is_Flagged_Row'] = out.index.isin(flagged_idx)
    front = ['Case_IDs', 'Is_Flagged_Row', 'Flag_Reason', 'Date', 'Ticker',
             'Open', 'High', 'Low', 'Close', 'Volume', 'Diag_Return_Pct']
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]

print('✅ Helper functions defined: scan_extremes(), build_context()')


✅ Helper functions defined: scan_extremes(), build_context()


## Step 3a: Apply Ticker-Level Cleaning (Optional)

If **`ticker_level_cleaning.csv`** is present, apply manual per-ticker deletions decided after reviewing the extreme-cases report. Otherwise skip.

Actions supported:
- `remove_ticker` — drop the entire ticker
- `delete_before` — drop rows strictly before `Date_Value` (the named date is **kept**)
- `delete_dates` — drop only the specific comma-separated date(s)

Runs on RAW data, before the extreme-cases diagnostic, so the diagnostic reflects the post-ticker-cleaning state.

In [5]:
import os

print("Step 3a: Applying ticker-level cleaning (if file provided)...")
print("-"*80)

# Snapshot RAW state BEFORE ticker cleaning (for the report, stage 1)
df_raw_snapshot = df.copy()
print(f"📸 Raw snapshot stored: {len(df_raw_snapshot):,} rows, {df_raw_snapshot['Ticker'].nunique()} tickers")

ticker_clean_log = []  # records what each instruction removed

if os.path.exists(TICKER_CLEANING_FILE):
    tcdf = pd.read_csv(TICKER_CLEANING_FILE, dtype=str).fillna('')
    print(f"\n✅ Found {TICKER_CLEANING_FILE} with {len(tcdf)} instruction(s)")

    rows_before_all = len(df)
    for _, instr in tcdf.iterrows():
        tkr = instr['Ticker'].strip()
        action = instr['Action'].strip().lower()
        date_value = instr['Date_Value'].strip()

        mask_ticker = df['Ticker'] == tkr
        n_ticker_rows = int(mask_ticker.sum())
        if n_ticker_rows == 0:
            ticker_clean_log.append({'Ticker': tkr, 'Action': action,
                'Date_Value': date_value, 'Rows_Removed': 0,
                'Note': 'Ticker not found in data'})
            print(f"  ⚠️  {tkr}: not found in data, skipped")
            continue

        if action == 'remove_ticker':
            to_drop = mask_ticker
        elif action == 'delete_before':
            cutoff = pd.to_datetime(date_value)
            to_drop = mask_ticker & (df['Date'] < cutoff)
        elif action == 'delete_dates':
            target_dates = [pd.to_datetime(x.strip()).normalize()
                            for x in date_value.split(',') if x.strip()]
            to_drop = mask_ticker & (df['Date'].dt.normalize().isin(target_dates))
        else:
            ticker_clean_log.append({'Ticker': tkr, 'Action': action,
                'Date_Value': date_value, 'Rows_Removed': 0,
                'Note': 'Unknown action - skipped'})
            print(f"  ⚠️  {tkr}: unknown action '{action}', skipped")
            continue

        n_removed = int(to_drop.sum())
        df = df[~to_drop].copy()
        ticker_clean_log.append({'Ticker': tkr, 'Action': action,
            'Date_Value': date_value, 'Rows_Removed': n_removed, 'Note': 'OK'})
        print(f'  OK {tkr} {action}: removed {n_removed} rows')

    total_removed = rows_before_all - len(df)
    print(f"\n📊 Ticker-level cleaning complete:")
    print(f"   Rows removed: {total_removed:,}")
    print(f"   Rows remaining: {len(df):,}")
    print(f"   Tickers remaining: {df['Ticker'].nunique()}")
else:
    print(f"\nℹ️  {TICKER_CLEANING_FILE} not found — skipping ticker-level cleaning.")

# Snapshot state AFTER ticker cleaning (for the report, stage 2)
df_after_ticker_snapshot = df.copy()
ticker_clean_log_df = pd.DataFrame(ticker_clean_log) if ticker_clean_log else pd.DataFrame(
    columns=['Ticker','Action','Date_Value','Rows_Removed','Note'])
print("-"*80)


Step 3a: Applying ticker-level cleaning (if file provided)...
--------------------------------------------------------------------------------
📸 Raw snapshot stored: 2,449,347 rows, 575 tickers

✅ Found ticker_level_cleaning.csv with 36 instruction(s)
  OK PSUBNKBEES remove_ticker: removed 4300 rows
  OK GOLDBEES remove_ticker: removed 4299 rows
  OK BANKBEES remove_ticker: removed 4300 rows
  OK LIQUIDBEES remove_ticker: removed 4300 rows
  OK JUNIORBEES remove_ticker: removed 4300 rows
  OK THYROCARE delete_dates: removed 1 rows
  OK NIFTYBEES delete_dates: removed 1 rows
  OK NIFTYBEES delete_dates: removed 2 rows
  OK SETFGOLD delete_before: removed 401 rows
  OK SETFGOLD delete_dates: removed 2 rows
  OK RAIN delete_before: removed 292 rows
  OK ADANIENT delete_before: removed 2072 rows
  OK BAJAJFINSV delete_before: removed 345 rows
  OK UNITECH delete_dates: removed 1 rows
  OK GVT&D delete_before: removed 370 rows
  OK PCJEWELLER delete_dates: removed 2 rows
  OK TIMKEN delete_

## Step 3b: Pre-Cleaning Diagnostic — Extreme Moves & Bad Prices

**Runs BEFORE any cleaning.** Flags rows where:
- Daily return < `DIAG_LOWER_PCT`% (default -50%) or > `DIAG_UPPER_PCT`% (default +50%)
- Close is NaN, zero, or negative

For every flagged row, exports a context window of **10 rows before and 10 rows after** (same ticker) so you can eyeball whether the move is a real event (split / bonus / dividend) or a data error.

Output: **`precleaning_extreme_cases.csv`** — does NOT modify `df`.

In [6]:
# ============================================================================
# PRE-CLEANING DIAGNOSTIC (uses shared helpers; does not modify df)
# Runs on the POST-ticker-cleaning data (df_after_ticker_snapshot).
# ============================================================================
DIAG_OUTPUT_FILE = 'precleaning_extreme_cases.csv'

print("Step 3b: Pre-cleaning diagnostic for extreme moves & bad prices...")
print("-"*80)
print(f"   Return flags : < {DIAG_LOWER_PCT}%  or  > {DIAG_UPPER_PCT}%")
print(f"   Price flags  : NaN, zero, or negative Close")
print(f"   Context      : {DIAG_CONTEXT_ROWS} rows before & after each case")
print()

# Scan + context on the current df (already ticker-cleaned)
_d, _mask, _summary = scan_extremes(df, DIAG_UPPER_PCT, DIAG_LOWER_PCT)
n_flagged = int(_mask.sum())
print(f"🔍 Flagged rows found: {n_flagged:,}")
if n_flagged > 0:
    print(f"   Tickers affected: {_summary.shape[0]}")
    ctx = build_context(_d, _mask, DIAG_CONTEXT_ROWS)
    ctx.to_csv(DIAG_OUTPUT_FILE, index=False)
    print(f"\n✅ Saved {DIAG_OUTPUT_FILE}")
    print(f"   Cases: {n_flagged:,}  |  Rows exported (incl. context): {len(ctx):,}")
    print(f"\nPreview (first 15 rows):")
    display(ctx.head(15))
else:
    print("\n✅ No extreme moves or bad prices found - nothing to export.")
print("-"*80)


Step 3b: Pre-cleaning diagnostic for extreme moves & bad prices...
--------------------------------------------------------------------------------
   Return flags : < -50.0%  or  > 50.0%
   Price flags  : NaN, zero, or negative Close
   Context      : 10 rows before & after each case

🔍 Flagged rows found: 39
   Tickers affected: 37

✅ Saved precleaning_extreme_cases.csv
   Cases: 39  |  Rows exported (incl. context): 752

Preview (first 15 rows):


,Case_IDs,Is_Flagged_Row,Flag_Reason,Date,Ticker,Open,High,Low,Close,Volume,Diag_Return_Pct,Value_Traded,Value_Traded_Cr
3639,1,False,,2021-10-07,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3640,1,False,,2021-10-08,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3641,1,False,,2021-10-11,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3642,1,False,,2021-10-12,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3643,1,False,,2021-10-13,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3644,1,False,,2021-10-14,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3645,1,False,,2021-10-18,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3646,1,False,,2021-10-19,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3647,1,False,,2021-10-20,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000
3648,1,False,,2021-10-21,3IINFOLTD,84.500000,84.500000,84.500000,84.500000,0,0.000000,0.000000e+00,0.000000


--------------------------------------------------------------------------------


## Step 4: Initial Data Quality Assessment

Check for missing values and data quality issues **BEFORE** cleaning.

In [7]:
print("="*80)
print("INITIAL DATA QUALITY ASSESSMENT")
print("="*80)

# Check missing values
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percent': (df.isnull().sum().values / len(df) * 100).round(2)
})

print("\n📊 MISSING VALUES BEFORE CLEANING:")
display(missing_summary[missing_summary['Missing_Count'] > 0])

print(f"\n✅ Total missing values: {df.isnull().sum().sum():,}")
print(f"   Missing value percentage: {(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100):.2f}%")

print("-"*80)

INITIAL DATA QUALITY ASSESSMENT

📊 MISSING VALUES BEFORE CLEANING:


,Column,Missing_Count,Missing_Percent



✅ Total missing values: 0
   Missing value percentage: 0.00%
--------------------------------------------------------------------------------


## Step 5: Handle Missing Values in Price Columns

**Strategy:**
- Forward fill small gaps (≤3 days)
- Interpolate medium gaps (4-5 days)
- Remove stocks with large gaps (>5 consecutive days)

In [8]:
print("Step 1: Handling missing values in price columns...")
print("-"*80)

price_columns = ['Open', 'High', 'Low', 'Close']

print(f"📊 BEFORE HANDLING MISSING PRICES:")
print(f"   Total records: {len(df):,}")
for col in price_columns:
    missing = df[col].isnull().sum()
    print(f"   Missing {col}: {missing} ({missing/len(df)*100:.2f}%)")

# Sort by Ticker and Date for proper forward fill
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# Apply filling strategy per ticker
def fill_missing_prices(group):
    """
    Fill missing prices using forward fill and interpolation
    """
    # Forward fill for small gaps (up to 3 days)
    group[price_columns] = group[price_columns].fillna(method='ffill', limit=3)

    # Linear interpolation for remaining gaps (up to 5 days)
    group[price_columns] = group[price_columns].interpolate(method='linear', limit=5)

    return group

# Apply per ticker
df = df.groupby('Ticker', group_keys=False).apply(fill_missing_prices)

print(f"\n✅ After filling:")
for col in price_columns:
    missing = df[col].isnull().sum()
    print(f"   Missing {col}: {missing} ({missing/len(df)*100:.2f}%)")

# Remove rows that still have missing prices (large gaps)
rows_before = len(df)
df = df.dropna(subset=price_columns)
rows_removed = rows_before - len(df)

print(f"\n📊 AFTER HANDLING MISSING PRICES:")
print(f"   Total records: {len(df):,}")
print(f"   Rows removed: {rows_removed:,} ({rows_removed/rows_before*100:.2f}%)")
print(f"   Remaining missing in price columns: {df[price_columns].isnull().sum().sum()}")
print("-"*80)

Step 1: Handling missing values in price columns...
--------------------------------------------------------------------------------
📊 BEFORE HANDLING MISSING PRICES:
   Total records: 2,418,551
   Missing Open: 0 (0.00%)
   Missing High: 0 (0.00%)
   Missing Low: 0 (0.00%)
   Missing Close: 0 (0.00%)

✅ After filling:
   Missing Open: 0 (0.00%)
   Missing High: 0 (0.00%)
   Missing Low: 0 (0.00%)
   Missing Close: 0 (0.00%)

📊 AFTER HANDLING MISSING PRICES:
   Total records: 2,418,551
   Rows removed: 0 (0.00%)
   Remaining missing in price columns: 0
--------------------------------------------------------------------------------


## Step 6: Handle Missing Values in Volume

**Strategy:** Volume = 0 likely means no trading, which is valid. Fill NaN with 0.

In [9]:
print("Step 2: Handling missing values in Volume...")
print("-"*80)

print(f"📊 BEFORE:")
missing_volume = df['Volume'].isnull().sum()
print(f"   Missing Volume: {missing_volume} ({missing_volume/len(df)*100:.2f}%)")

# Fill missing volume with 0 (no trading)
df['Volume'] = df['Volume'].fillna(0)

# Recalculate Value_Traded after volume fix (if it exists)
if 'Value_Traded' in df.columns:
    df['Value_Traded'] = df['Close'] * df['Volume']
    df['Value_Traded_Cr'] = df['Value_Traded'] / 10000000

print(f"\n✅ After filling with 0:")
missing_volume = df['Volume'].isnull().sum()
print(f"   Missing Volume: {missing_volume}")
print("-"*80)

Step 2: Handling missing values in Volume...
--------------------------------------------------------------------------------
📊 BEFORE:
   Missing Volume: 0 (0.00%)

✅ After filling with 0:
   Missing Volume: 0
--------------------------------------------------------------------------------


## Step 7: Analyze Return Distribution

**Before removing outliers, understand the return distribution.**

**Using Close for return calculations** (split-adjusted).

In [10]:
print("="*80)
print("RETURN DISTRIBUTION ANALYSIS")
print("="*80)

# Calculate daily returns using Close (split-adjusted)
df['Daily_Return'] = df.groupby('Ticker')['Close'].pct_change()

# Remove NaN for analysis
returns = df['Daily_Return'].dropna()

print(f"\n📊 RETURN STATISTICS (using Close):")
print(f"   Total return observations: {len(returns):,}")
print(f"   Mean: {returns.mean()*100:.4f}%")
print(f"   Std Dev: {returns.std()*100:.4f}%")
print(f"   Min: {returns.min()*100:.2f}%")
print(f"   Max: {returns.max()*100:.2f}%")

print(f"\n📈 PERCENTILES:")
percentiles = [0.1, 1, 5, 25, 50, 75, 95, 99, 99.9]
for p in percentiles:
    val = returns.quantile(p/100)
    print(f"   {p:5.1f}%: {val*100:8.2f}%")

print(f"\n🎯 EXTREME RETURNS ANALYSIS:")
extreme_categories = [
    ('+20%', 0.20, 'upper'),
    ('+50%', 0.50, 'upper'),
    ('+100%', 1.00, 'upper'),
    ('+200%', 2.00, 'upper'),
    ('-20%', -0.20, 'lower'),
    ('-50%', -0.50, 'lower'),
    ('-80%', -0.80, 'lower'),
]

for label, threshold, direction in extreme_categories:
    if direction == 'upper':
        count = (returns > threshold).sum()
    else:
        count = (returns < threshold).sum()
    pct = count / len(returns) * 100
    print(f"   Returns > {label}: {count:6,} ({pct:.4f}%)")

print(f"\n💡 OUR THRESHOLD IMPACT:")
extreme_positive = (returns > UPPER_RETURN_THRESHOLD).sum()
extreme_negative = (returns < LOWER_RETURN_THRESHOLD).sum()
total_extreme = extreme_positive + extreme_negative
print(f"   Using +{UPPER_RETURN_THRESHOLD*100:.0f}%/{LOWER_RETURN_THRESHOLD*100:.0f}% threshold will remove:")
print(f"   - Positive outliers (>{UPPER_RETURN_THRESHOLD*100:.0f}%): {extreme_positive:,}")
print(f"   - Negative outliers (<{LOWER_RETURN_THRESHOLD*100:.0f}%): {extreme_negative:,}")
print(f"   - Total: {total_extreme:,} records ({total_extreme/len(returns)*100:.4f}%)")

print("\n" + "="*80)

RETURN DISTRIBUTION ANALYSIS

📊 RETURN STATISTICS (using Close):
   Total return observations: 2,417,981
   Mean: 0.1017%
   Std Dev: 3.3093%
   Min: -95.16%
   Max: 436.36%

📈 PERCENTILES:
     0.1%:   -15.80%
     1.0%:    -8.28%
     5.0%:    -4.52%
    25.0%:    -1.48%
    50.0%:    -0.02%
    75.0%:     1.45%
    95.0%:     5.26%
    99.0%:    10.39%
    99.9%:    20.76%

🎯 EXTREME RETURNS ANALYSIS:
   Returns > +20%:  2,879 (0.1191%)
   Returns > +50%:     18 (0.0007%)
   Returns > +100%:      7 (0.0003%)
   Returns > +200%:      3 (0.0001%)
   Returns > -20%:    792 (0.0328%)
   Returns > -50%:     21 (0.0009%)
   Returns > -80%:      2 (0.0001%)

💡 OUR THRESHOLD IMPACT:
   Using +100%/-80% threshold will remove:
   - Positive outliers (>100%): 7
   - Negative outliers (<-80%): 2
   - Total: 9 records (0.0004%)



## Step 8: Remove Outliers Using Absolute Thresholds

**Using Close for return calculations.**

Remove records with extreme returns using **absolute thresholds** (±100%/-80%).

In [11]:
print("Step 3: Detecting and handling outliers using ABSOLUTE thresholds...")
print("-"*80)

rows_before = len(df)

print(f"📊 BEFORE OUTLIER REMOVAL:")
print(f"   Total records: {rows_before:,}")

# Daily returns already calculated in Step 7 using Close
# Mark outliers
df['Is_Outlier'] = (
    (df['Daily_Return'] > UPPER_RETURN_THRESHOLD) |
    (df['Daily_Return'] < LOWER_RETURN_THRESHOLD)
)

# Count outliers
outlier_count = df['Is_Outlier'].sum()
positive_outliers = (df['Daily_Return'] > UPPER_RETURN_THRESHOLD).sum()
negative_outliers = (df['Daily_Return'] < LOWER_RETURN_THRESHOLD).sum()

print(f"\n🔍 OUTLIERS DETECTED:")
print(f"   Threshold: >{UPPER_RETURN_THRESHOLD*100:.0f}% or <{LOWER_RETURN_THRESHOLD*100:.0f}%")
print(f"   Total outliers: {outlier_count:,} ({outlier_count/rows_before*100:.4f}%)")
print(f"   - Positive (>{UPPER_RETURN_THRESHOLD*100:.0f}%): {positive_outliers:,}")
print(f"   - Negative (<{LOWER_RETURN_THRESHOLD*100:.0f}%): {negative_outliers:,}")

# Show examples of outliers
if outlier_count > 0:
    print(f"\n📋 SAMPLE OUTLIERS:")
    outlier_samples = df[df['Is_Outlier']][['Date', 'Ticker', 'Close', 'Daily_Return']].copy()
    outlier_samples['Daily_Return_Pct'] = outlier_samples['Daily_Return'] * 100
    outlier_samples = outlier_samples.sort_values('Daily_Return', ascending=False)

    print(f"\n   Top 5 positive outliers:")
    display(outlier_samples.head(5)[['Date', 'Ticker', 'Close', 'Daily_Return_Pct']])

    print(f"\n   Top 5 negative outliers:")
    display(outlier_samples.tail(5)[['Date', 'Ticker', 'Close', 'Daily_Return_Pct']])

# Remove outliers
df = df[~df['Is_Outlier']].copy()
df = df.drop('Is_Outlier', axis=1)

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"\n📊 AFTER OUTLIER REMOVAL:")
print(f"   Total records: {rows_after:,}")
print(f"   Rows removed: {rows_removed:,} ({rows_removed/rows_before*100:.4f}%)")
print("-"*80)

Step 3: Detecting and handling outliers using ABSOLUTE thresholds...
--------------------------------------------------------------------------------
📊 BEFORE OUTLIER REMOVAL:
   Total records: 2,418,551

🔍 OUTLIERS DETECTED:
   Threshold: >100% or <-80%
   Total outliers: 9 (0.0004%)
   - Positive (>100%): 7
   - Negative (<-80%): 2

📋 SAMPLE OUTLIERS:

   Top 5 positive outliers:


,Date,Ticker,Close,Daily_Return_Pct
103398,2020-02-19,ALOKINDS,17.700001,436.363667
1607880,2020-11-03,ORCHPHARMA,17.150000,214.678903
2168251,2020-11-14,THYROCARE,947.747206,202.819652
1268538,2010-06-24,KIRLOSENG,290.433216,129.489371
708912,2018-06-25,FCL,5.807489,113.644509



   Top 5 negative outliers:


,Date,Ticker,Close,Daily_Return_Pct
708912,2018-06-25,FCL,5.807489,113.644509
291867,2025-07-14,BCG,21.799999,112.682919
1672455,2011-09-27,PGEL,44.965511,109.302329
55143,2015-06-04,ADANIENT,57.473855,-80.463981
1630975,2020-01-27,PATANJALI,5.268179,-95.164179



📊 AFTER OUTLIER REMOVAL:
   Total records: 2,418,542
   Rows removed: 9 (0.0004%)
--------------------------------------------------------------------------------


## Step 9: Fix Data Quality Issues

Remove records with invalid data.

In [12]:
print("Step 4: Fixing data quality issues...")
print("-"*80)

rows_before = len(df)

print(f"📊 BEFORE QUALITY FIXES:")
print(f"   Total records: {rows_before:,}")

# Check for various quality issues
zero_prices = (df['Close'] <= 0).sum()
high_low_violation = (df['High'] < df['Low']).sum()
close_outside = ((df['Close'] < df['Low']) | (df['Close'] > df['High'])).sum()
open_outside = ((df['Open'] < df['Low']) | (df['Open'] > df['High'])).sum()

print(f"   Zero/negative prices: {zero_prices}")
print(f"   High < Low violations: {high_low_violation}")
print(f"   Close outside [Low, High]: {close_outside}")
print(f"   Open outside [Low, High]: {open_outside}")

# Remove all quality issues
df = df[
    (df['Close'] > 0) &
    (df['High'] >= df['Low']) &
    (df['Close'] >= df['Low']) &
    (df['Close'] <= df['High']) &
    (df['Open'] >= df['Low']) &
    (df['Open'] <= df['High'])
].copy()

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"\n✅ Quality issues fixed!")
print(f"\n📊 AFTER QUALITY FIXES:")
print(f"   Total records: {rows_after:,}")
print(f"   Rows removed: {rows_removed:,} ({rows_removed/rows_before*100:.4f}%)")
print("-"*80)

Step 4: Fixing data quality issues...
--------------------------------------------------------------------------------
📊 BEFORE QUALITY FIXES:
   Total records: 2,418,542
   Zero/negative prices: 0
   High < Low violations: 6
   Close outside [Low, High]: 14
   Open outside [Low, High]: 875

✅ Quality issues fixed!

📊 AFTER QUALITY FIXES:
   Total records: 2,417,660
   Rows removed: 882 (0.0365%)
--------------------------------------------------------------------------------


## Step 10: Final Missing Values Check

In [13]:
print("="*80)
print("FINAL MISSING VALUES CHECK")
print("="*80)

missing_summary_final = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percent': (df.isnull().sum().values / len(df) * 100).round(4)
})

print("\n📊 MISSING VALUES AFTER CLEANING:")
display(missing_summary_final)

total_missing = df.isnull().sum().sum()
print(f"\n✅ Total missing values: {total_missing:,}")

if total_missing == 0:
    print("\n🎉 Perfect! No missing values!")
else:
    missing_cols = missing_summary_final[missing_summary_final['Missing_Count'] > 0]
    print(f"\n⚠️  {total_missing} missing values in:")
    for _, row in missing_cols.iterrows():
        print(f"   - {row['Column']}: {row['Missing_Count']} ({row['Missing_Percent']:.4f}%)")
    print(f"\n💡 Note: Missing in Daily_Return is normal for first record of each stock.")

print("-"*80)

FINAL MISSING VALUES CHECK

📊 MISSING VALUES AFTER CLEANING:


,Column,Missing_Count,Missing_Percent
0,Date,0,0.0000
1,Open,0,0.0000
2,High,0,0.0000
3,Low,0,0.0000
4,Close,0,0.0000
5,Volume,0,0.0000
6,Ticker,0,0.0000
7,Value_Traded,0,0.0000
8,Value_Traded_Cr,0,0.0000
9,Daily_Return,569,0.0235



✅ Total missing values: 569

⚠️  569 missing values in:
   - Daily_Return: 569 (0.0235%)

💡 Note: Missing in Daily_Return is normal for first record of each stock.
--------------------------------------------------------------------------------


## Step 11: Remove Stocks with Insufficient Data

In [14]:
print("Step 5: Removing stocks with insufficient data after cleaning...")
print("-"*80)

stocks_before = df['Ticker'].nunique()
records_before = len(df)

print(f"📊 BEFORE:")
print(f"   Unique stocks: {stocks_before}")
print(f"   Total records: {records_before:,}")

# Count records per ticker
ticker_counts = df.groupby('Ticker').size()

# Keep only tickers with >= 250 days
min_days_threshold = 250
valid_tickers = ticker_counts[ticker_counts >= min_days_threshold].index
df = df[df['Ticker'].isin(valid_tickers)].copy()

stocks_after = df['Ticker'].nunique()
records_after = len(df)

print(f"\n✅ Stocks with ≥{min_days_threshold} trading days: {stocks_after}")
print(f"❌ Stocks removed: {stocks_before - stocks_after}")

print(f"\n📊 AFTER:")
print(f"   Unique stocks: {stocks_after}")
print(f"   Total records: {records_after:,}")
print(f"   Records removed: {records_before - records_after:,}")
print("-"*80)

Step 5: Removing stocks with insufficient data after cleaning...
--------------------------------------------------------------------------------
📊 BEFORE:
   Unique stocks: 570
   Total records: 2,417,660

✅ Stocks with ≥250 trading days: 570
❌ Stocks removed: 0

📊 AFTER:
   Unique stocks: 570
   Total records: 2,417,660
   Records removed: 0
--------------------------------------------------------------------------------


## Step 11b: Build Detailed Multi-Tab Excel Report

Captures the extreme-move / bad-price picture at **three stages** using the report thresholds (±50% by default, independent of the ±100%/−80% outlier-removal thresholds):

1. **Raw** (as loaded) — `df_raw_snapshot`
2. **After ticker-level cleaning** — `df_after_ticker_snapshot`
3. **After all other processing** — current `df`

Each stage gets a totals tab + a ±10-row context tab. Plus a summary tab and the ticker-cleaning log.

**Note:** At stage 3, '10 rows before/after' means 10 *surviving* rows (calendar gaps exist where rows were removed).

In [15]:
print("Step 11b: Building detailed multi-tab Excel report...")
print("-"*80)
print("This may take a minute or two (three context scans on full data).\n")

# --- Run scans at all three stages ---
print("  Scanning Stage 1 (raw)...")
raw_d, raw_mask, raw_summary = scan_extremes(df_raw_snapshot, DIAG_UPPER_PCT, DIAG_LOWER_PCT)
raw_ctx = build_context(raw_d, raw_mask, DIAG_CONTEXT_ROWS)

print("  Scanning Stage 2 (after ticker cleaning)...")
tc_d, tc_mask, tc_summary = scan_extremes(df_after_ticker_snapshot, DIAG_UPPER_PCT, DIAG_LOWER_PCT)
tc_ctx = build_context(tc_d, tc_mask, DIAG_CONTEXT_ROWS)

print("  Scanning Stage 3 (after all processing)...")
fin_d, fin_mask, fin_summary = scan_extremes(df, DIAG_UPPER_PCT, DIAG_LOWER_PCT)
fin_ctx = build_context(fin_d, fin_mask, DIAG_CONTEXT_ROWS)

# --- Build summary tab ---
def stage_totals(summary, d, mask, label, n_rows, n_tickers):
    return {
        'Stage': label,
        'Total_Rows': n_rows,
        'Total_Tickers': n_tickers,
        'Flagged_Rows': int(mask.sum()),
        'Tickers_With_Flags': int(summary.shape[0]),
        'Returns_Above_Upper': int(summary['Count_Return_Above'].sum()) if len(summary) else 0,
        'Returns_Below_Lower': int(summary['Count_Return_Below'].sum()) if len(summary) else 0,
        'Close_NaN': int(summary['Count_Close_NaN'].sum()) if len(summary) else 0,
        'Close_LE0': int(summary['Count_Close_LE0'].sum()) if len(summary) else 0,
    }

summary_rows = [
    stage_totals(raw_summary, raw_d, raw_mask, '1_Raw',
                 len(df_raw_snapshot), df_raw_snapshot['Ticker'].nunique()),
    stage_totals(tc_summary, tc_d, tc_mask, '2_After_TickerClean',
                 len(df_after_ticker_snapshot), df_after_ticker_snapshot['Ticker'].nunique()),
    stage_totals(fin_summary, fin_d, fin_mask, '3_After_Processing',
                 len(df), df['Ticker'].nunique()),
]
summary_tab = pd.DataFrame(summary_rows)

# Add threshold + config context as a second small block
config_tab = pd.DataFrame({
    'Setting': ['Report_Upper_Pct', 'Report_Lower_Pct', 'Context_Rows',
                'Outlier_Upper_Removal', 'Outlier_Lower_Removal', 'Min_Days_Threshold'],
    'Value': [DIAG_UPPER_PCT, DIAG_LOWER_PCT, DIAG_CONTEXT_ROWS,
              UPPER_RETURN_THRESHOLD*100, LOWER_RETURN_THRESHOLD*100, 250]
})

# --- Write Excel with multiple tabs ---
print(f"\n  Writing {CLEANING_REPORT_XLSX}...")
# Pick an available Excel engine (xlsxwriter preferred, openpyxl fallback)
try:
    import xlsxwriter  # noqa
    _xl_engine = 'xlsxwriter'
except ModuleNotFoundError:
    _xl_engine = 'openpyxl'
print(f'   Using Excel engine: {_xl_engine}')
with pd.ExcelWriter(CLEANING_REPORT_XLSX, engine=_xl_engine) as writer:
    summary_tab.to_excel(writer, sheet_name='0_Summary', index=False, startrow=0)
    config_tab.to_excel(writer, sheet_name='0_Summary', index=False, startrow=len(summary_tab)+3)
    ticker_clean_log_df.to_excel(writer, sheet_name='4_TickerCleaning_Log', index=False)

    raw_summary.to_excel(writer, sheet_name='1a_Raw_Totals', index=False)
    (raw_ctx if len(raw_ctx) else pd.DataFrame({'Note':['No flagged rows']})
     ).to_excel(writer, sheet_name='1b_Raw_Context', index=False)

    tc_summary.to_excel(writer, sheet_name='2a_AfterTicker_Totals', index=False)
    (tc_ctx if len(tc_ctx) else pd.DataFrame({'Note':['No flagged rows']})
     ).to_excel(writer, sheet_name='2b_AfterTicker_Context', index=False)

    fin_summary.to_excel(writer, sheet_name='3a_AfterProc_Totals', index=False)
    (fin_ctx if len(fin_ctx) else pd.DataFrame({'Note':['No flagged rows']})
     ).to_excel(writer, sheet_name='3b_AfterProc_Context', index=False)

    # Light formatting: freeze header row on each sheet
    for sheet in writer.sheets.values():
        sheet.freeze_panes(1, 0)

print(f"✅ Saved {CLEANING_REPORT_XLSX}")
print(f"\n📊 Stage comparison:")
display(summary_tab)
print("-"*80)


Step 11b: Building detailed multi-tab Excel report...
--------------------------------------------------------------------------------
This may take a minute or two (three context scans on full data).

  Scanning Stage 1 (raw)...
  Scanning Stage 2 (after ticker cleaning)...
  Scanning Stage 3 (after all processing)...

  Writing cleaning_report.xlsx...
   Using Excel engine: xlsxwriter
✅ Saved cleaning_report.xlsx

📊 Stage comparison:


,Stage,Total_Rows,Total_Tickers,Flagged_Rows,Tickers_With_Flags,Returns_Above_Upper,Returns_Below_Lower,Close_NaN,Close_LE0
0,1_Raw,2449347,575,1338,68,65,53,0,1236
1,2_After_TickerClean,2418551,570,39,37,18,21,0,0
2,3_After_Processing,2417660,570,37,36,17,20,0,0


--------------------------------------------------------------------------------


## Step 12: Overall Cleaning Summary

In [16]:
print("="*80)
print("COMPLETE CLEANING SUMMARY")
print("="*80)

cleaning_summary = pd.DataFrame([
    {'Stage': '1. Initial (After Code 1b)', 'Stocks': initial_stocks, 'Records': initial_records},
    {'Stage': '2. After handling missing prices', 'Stocks': df['Ticker'].nunique(), 'Records': len(df)},
    {'Stage': '3. After outlier removal', 'Stocks': df['Ticker'].nunique(), 'Records': len(df)},
    {'Stage': '4. After quality fixes', 'Stocks': df['Ticker'].nunique(), 'Records': len(df)},
    {'Stage': '5. Final (After all cleaning)', 'Stocks': df['Ticker'].nunique(), 'Records': len(df)}
])

display(cleaning_summary)

print(f"\n📊 OVERALL IMPACT:")
print(f"   Initial stocks: {initial_stocks}")
print(f"   Final stocks: {df['Ticker'].nunique()}")
print(f"   Stocks removed: {initial_stocks - df['Ticker'].nunique()}")
print(f"   Stock retention: {df['Ticker'].nunique() / initial_stocks * 100:.2f}%")
print(f"\n   Initial records: {initial_records:,}")
print(f"   Final records: {len(df):,}")
print(f"   Records removed: {initial_records - len(df):,}")
print(f"   Record retention: {len(df) / initial_records * 100:.2f}%")

print("\n" + "="*80)

COMPLETE CLEANING SUMMARY


,Stage,Stocks,Records
0,1. Initial (After Code 1b),575,2449347
1,2. After handling missing prices,570,2417660
2,3. After outlier removal,570,2417660
3,4. After quality fixes,570,2417660
4,5. Final (After all cleaning),570,2417660



📊 OVERALL IMPACT:
   Initial stocks: 575
   Final stocks: 570
   Stocks removed: 5
   Stock retention: 99.13%

   Initial records: 2,449,347
   Final records: 2,417,660
   Records removed: 31,687
   Record retention: 98.71%



## Step 13: Save Output Files

In [17]:
print("="*80)
print("SAVING OUTPUT FILES")
print("="*80)

# Save cleaned dataset
print(f"\nSaving {OUTPUT_FILE}...")
df.to_csv(OUTPUT_FILE, index=False)
file_size_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"✅ Saved: {OUTPUT_FILE}")
print(f"   Size: {file_size_mb:.2f} MB")
print(f"   Rows: {len(df):,}")
print(f"   Stocks: {df['Ticker'].nunique()}")
print(f"   Columns: {list(df.columns)}")

# Save cleaning report
print(f"\nSaving {CLEANING_REPORT_FILE}...")
cleaning_summary.to_csv(CLEANING_REPORT_FILE, index=False)
print(f"✅ Saved: {CLEANING_REPORT_FILE}")

# Detailed multi-tab Excel report was already written in Step 11b
import os as _os
if _os.path.exists(CLEANING_REPORT_XLSX):
    print(f"✅ Detailed report present: {CLEANING_REPORT_XLSX}")

print("\n" + "="*80)
print("✅ ALL FILES SAVED!")
print("="*80)

SAVING OUTPUT FILES

Saving data_clean.csv...
✅ Saved: data_clean.csv
   Size: 313.91 MB
   Rows: 2,417,660
   Stocks: 570
   Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Value_Traded', 'Value_Traded_Cr', 'Daily_Return']

Saving cleaning_report.csv...
✅ Saved: cleaning_report.csv
✅ Detailed report present: cleaning_report.xlsx

✅ ALL FILES SAVED!


## Step 14: Final Summary

In [18]:
print("="*80)
print("FINAL SUMMARY - CODE 2 (V2)")
print("="*80)

print(f"""
📊 Output Dataset (data_clean.csv):
   - Total stocks: {df['Ticker'].nunique()}
   - Total records: {len(df):,}
   - Date range: {df['Date'].min()} to {df['Date'].max()}
   - Missing values: {df.isnull().sum().sum()}

✅ Cleaning Steps Completed:
   1. Handled missing values in prices
   2. Handled missing volume
   3. Analyzed return distribution
   4. Removed outliers (±{UPPER_RETURN_THRESHOLD*100:.0f}%/{LOWER_RETURN_THRESHOLD*100:.0f}%)
   5. Fixed data quality issues
   6. Removed stocks with insufficient data

💡 KEY METHODOLOGY:
   - Using Close (split-adjusted) for all calculations
   - No Adj_Close column
   - Absolute threshold outlier detection
   - All returns calculated using Close

📁 Output Files:
   1. {OUTPUT_FILE} - Clean dataset
   2. {CLEANING_REPORT_FILE} - Cleaning report

➡️  Next Step: Run Code 3 to add technical indicators and features
""")

print("="*80)

FINAL SUMMARY - CODE 2 (V2)

📊 Output Dataset (data_clean.csv):
   - Total stocks: 570
   - Total records: 2,417,660
   - Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   - Missing values: 569

✅ Cleaning Steps Completed:
   1. Handled missing values in prices
   2. Handled missing volume
   3. Analyzed return distribution
   4. Removed outliers (±100%/-80%)
   5. Fixed data quality issues
   6. Removed stocks with insufficient data

💡 KEY METHODOLOGY:
   - Using Close (split-adjusted) for all calculations
   - No Adj_Close column
   - Absolute threshold outlier detection
   - All returns calculated using Close

📁 Output Files:
   1. data_clean.csv - Clean dataset
   2. cleaning_report.csv - Cleaning report

➡️  Next Step: Run Code 3 to add technical indicators and features



## Step 15: Preview Clean Data

In [19]:
print("CLEAN DATA SAMPLE (First 20 rows):")
print("="*80)
display(df.head(20))

print("\nSUMMARY STATISTICS:")
print("="*80)
display(df.describe())

CLEAN DATA SAMPLE (First 20 rows):


,Date,Open,High,Low,Close,Volume,Ticker,Value_Traded,Value_Traded_Cr,Daily_Return
0,2007-01-02,962.630615,977.636194,858.319096,859.410422,965111,3IINFOLTD,8.294265e+08,82.942645,NaN
1,2007-01-03,983.365723,1002.645575,960.629995,976.272131,895672,3IINFOLTD,8.744196e+08,87.441961,0.135979
2,2007-01-04,964.904175,993.096474,954.900456,993.096474,247242,3IINFOLTD,2.455352e+08,24.553516,0.017233
3,2007-01-05,978.818542,999.007823,963.994804,963.994804,264776,3IINFOLTD,2.552427e+08,25.524269,-0.029304
4,2007-01-08,1023.380615,1032.202033,950.353414,976.272143,699857,3IINFOLTD,6.832509e+08,68.325089,0.012736
5,2007-01-09,994.733582,1052.482329,977.636270,1036.749161,423121,3IINFOLTD,4.386703e+08,43.867034,0.061947
6,2007-01-10,969.269470,994.460609,963.994738,989.913464,152130,3IINFOLTD,1.505955e+08,15.059554,-0.045176
7,2007-01-11,1062.940796,1076.582233,962.903593,977.636300,740662,3IINFOLTD,7.240981e+08,72.409806,-0.012402
8,2007-01-12,1064.941406,1095.861994,1031.565405,1086.312989,562895,3IINFOLTD,6.114801e+08,61.148015,0.111163
9,2007-01-15,1060.667114,1090.860113,1040.386803,1074.945106,283064,3IINFOLTD,3.042783e+08,30.427826,-0.010465



SUMMARY STATISTICS:


,Date,Open,High,Low,Close,Volume,Value_Traded,Value_Traded_Cr,Daily_Return
count,2417660,2.417660e+06,2.417660e+06,2.417660e+06,2.417660e+06,2.417660e+06,2.417660e+06,2.417660e+06,2.417091e+06
mean,2017-06-28 01:08:11.318712064,7.539122e+02,7.660268e+02,7.432629e+02,7.549984e+02,3.445840e+06,5.533455e+08,5.533455e+01,1.012956e-03
min,2007-01-02 00:00:00,2.000000e-01,2.329659e-01,1.500000e-01,2.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,-7.971906e-01
25%,2012-12-19 00:00:00,4.865000e+01,4.988459e+01,4.765328e+01,4.876591e+01,8.584500e+04,1.118711e+07,1.118711e+00,-1.476078e-02
50%,2017-11-10 00:00:00,1.460791e+02,1.492233e+02,1.433722e+02,1.463575e+02,4.007265e+05,6.875569e+07,6.875569e+00,-2.439537e-04
75%,2022-02-25 00:00:00,4.723353e+02,4.815001e+02,4.645434e+02,4.730940e+02,1.795159e+06,3.622788e+08,3.622788e+01,1.449281e-02
max,2026-06-12 00:00:00,1.622886e+05,1.635935e+05,1.602787e+05,1.619736e+05,2.179435e+09,3.766747e+11,3.766747e+04,9.014287e-01
std,NaN,3.761778e+03,3.808739e+03,3.720470e+03,3.767439e+03,1.752106e+07,1.942300e+09,1.942300e+02,3.287593e-02


## 📥 Download Files (Optional)

In [20]:
from google.colab import files

print("Downloading files...")
files.download(OUTPUT_FILE)
files.download(CLEANING_REPORT_FILE)

files.download(DIAG_OUTPUT_FILE)
files.download(CLEANING_REPORT_XLSX)
print("\n✅ Download complete!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!


---

## ✅ Code 2 Complete! (V2 - Using Close Only)

**Next Step:** Run **Code 3** to add technical indicators and derived features.

### What Changed in V2:
- ✅ Using Close (split-adjusted) instead of Adj_Close
- ✅ All return calculations use Close
- ✅ Absolute threshold outlier detection (±100%/-80%)
- ✅ Simpler methodology